# 04 — Model run & result tables

Four feature specifications (controls / +NLP / +traditional / full) for two outcomes
(cumulative funding, exit), across a gradient-boosted tree (nested CV) and a penalised-linear
model (repeated 10×5 CV). One-sided Nadeau–Bengio tests.

Input: `data/processed/2_final_sample_focus.csv`. Tables are shown inline and written to
`data/processed/ml_results/`.

In [1]:
import warnings, pickle
from pathlib import Path
import numpy as np, pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegressionCV, RidgeCV
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb, shap
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

SEED,N_SPLITS,N_SEEDS=42,5,10; SEEDS=[SEED+i for i in range(N_SEEDS)]; RHO=1/(N_SPLITS-1)
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data').exists())
PROC=ROOT/'data'/'processed'; OUT=PROC/'ml_results'; OUT.mkdir(parents=True,exist_ok=True)
P_FOCAL=PROC/'2_final_sample_focus.csv'
P_ROUNDS=ROOT/'data'/'raw'/'crunchbase'/'funding_rounds_crunchbase_export.csv'
WINNER_Q=0.90

In [2]:
TRAD=['bwd_cits_mean','npl_cits_mean','originality_mean','radicalness_mean','claims_mean','family_size_mean','patent_scope_mean']
NLP=['novelty_mean','competitive_density']; WINS=['bwd_cits_mean','npl_cits_mean','family_size_mean']
SPECS=('controls','nlp_only','baseline','+nlp'); CTRL_CAT=['country_region','tech_field_group']
CRMAP={'Germany':'DACH','Austria':'DACH','Switzerland':'DACH','Liechtenstein':'DACH','United Kingdom':'UK_Ireland','Ireland':'UK_Ireland','Sweden':'Nordic','Denmark':'Nordic','Norway':'Nordic','Finland':'Nordic','Iceland':'Nordic','The Netherlands':'Benelux','Belgium':'Benelux','Luxembourg':'Benelux','France':'France','Italy':'Southern','Spain':'Southern','Portugal':'Southern','Greece':'Southern'}
LABEL={'n_patents':'Portfolio size (n_patents)','family_size_mean':'Family size','novelty_mean':'Novelty','patent_scope_mean':'Patent scope','npl_cits_mean':'NPL citations','country_region':'Country region','competitive_density':'Competitive density','claims_mean':'Claims','radicalness_mean':'Radicalness','bwd_cits_mean':'Backward citations','originality_mean':'Originality','tech_field_group':'Technology field','founding_year':'Founding year','age_at_first_funding':'Age at first funding'}
def ftree(df,sp):
    f=['founding_year','n_patents','age_at_first_funding']+CTRL_CAT
    if sp in ('baseline','+nlp'): f+=TRAD+[c+'_missing' for c in TRAD if c+'_missing' in df.columns]
    if sp in ('nlp_only','+nlp'): f+=NLP
    return f
def flin(df,sp):
    oh=[c for c in df.columns if any(c.startswith(p+'_') for p in CTRL_CAT)]
    f=['founding_year','n_patents','age_at_first_funding']+oh
    if sp in ('baseline','+nlp'): f+=TRAD
    if sp in ('nlp_only','+nlp'): f+=NLP
    return f

In [3]:
def load():
    s=pd.read_csv(P_FOCAL); s=s[s.excluded_startup==0].copy()
    s['country_region']=s['country'].map(CRMAP).fillna('Other')
    for c in TRAD: s[c+'_missing']=s[c].isna().astype(int)   # keep flags; leave values NaN (tree fills 0 in tfit; linear median-imputes in winsor_impute)
    return s.reset_index(drop=True)
d=load()
small=(d['n_patents']<=3).values; life=(d['tech_field_group'].astype(str)=='life sciences').values
SAMPLE={'log_total_funding_usd':d[d.log_total_funding_usd.notna()].reset_index(drop=True),'exit_binary':d}
OUTCOMES={'log_total_funding_usd':('Cumulative funding','reg'),'exit_binary':('Exit','clf')}
MASK={'log_total_funding_usd':{'small':(SAMPLE['log_total_funding_usd']['n_patents']<=3).values,'life':(SAMPLE['log_total_funding_usd']['tech_field_group'].astype(str)=='life sciences').values},
      'exit_binary':{'small':small,'life':life}}
print('n(funding)=',len(SAMPLE['log_total_funding_usd']),' n(exit)=',len(SAMPLE['exit_binary']),' funding base(top decile)=0.10, exit base=',round(d.exit_binary.mean(),3))

n(funding)= 1435  n(exit)= 1810  funding base(top decile)=0.10, exit base= 0.205


In [4]:
GN={'C5':dict(n_estimators=500,learning_rate=0.02,max_depth=2,num_leaves=4,min_child_samples=50,reg_lambda=10.0,subsample=0.5,colsample_bytree=0.5,subsample_freq=1)}
_k=0
for md,nl in [(2,4),(3,8),(4,16)]:
    for lr,ne in [(0.01,800),(0.02,500),(0.05,300)]:
        for mcs,rl in [(20,1.0),(100,20.0)]:
            _k+=1; GN[f'G{_k:02d}']=dict(n_estimators=ne,learning_rate=lr,max_depth=md,num_leaves=nl,min_child_samples=mcs,reg_lambda=rl,subsample=0.7,colsample_bytree=0.7,subsample_freq=1)
def tfit(Xtr,ytr,Xte,kind,hp):
    Xtr,Xte=Xtr.copy(),Xte.copy()
    for c in TRAD+NLP+['age_at_first_funding']:
        if c in Xtr: Xtr[c]=Xtr[c].fillna(0); Xte[c]=Xte[c].fillna(0)
    if kind=='clf':
        m=lgb.LGBMClassifier(**hp,class_weight='balanced',random_state=SEED,verbose=-1); m.fit(Xtr,ytr); return m.predict_proba(Xte)[:,1],m
    m=lgb.LGBMRegressor(**hp,random_state=SEED,verbose=-1); m.fit(Xtr,ytr); return m.predict(Xte),m
def winsor_impute(Xtr,Xte,tt,te):
    Xtr,Xte=Xtr.reset_index(drop=True).copy(),Xte.reset_index(drop=True).copy(); tt,te=pd.Series(tt),pd.Series(te)
    for c in WINS:
        if c in Xtr:
            cap=Xtr[c].quantile(0.99)
            if pd.notna(cap): Xtr[c]=Xtr[c].clip(upper=cap); Xte[c]=Xte[c].clip(upper=cap)
    for c in [c for c in Xtr.columns if c in set(TRAD)|set(NLP)|{'age_at_first_funding'}]:
        if Xtr[c].isna().any() or Xte[c].isna().any():
            md=Xtr[c].groupby(tt).median(); g=Xtr[c].median(); Xtr[c]=Xtr[c].fillna(tt.map(md)).fillna(g); Xte[c]=Xte[c].fillna(te.map(md)).fillna(g)
    return Xtr,Xte
def nb_one(x):
    x=np.asarray(x,float); n=len(x); md,v=x.mean(),x.var(ddof=1); se=np.sqrt((1/n+RHO)*v); dd=n-1
    return dict(d=md, lb=md-stats.t.ppf(0.95,dd)*se, p=stats.t.sf(md/se if se>0 else 0,dd), se=se)
def stt(p): return '***' if p<0.01 else '**' if p<0.05 else '*' if p<0.10 else ''

In [5]:
def nested_tree(oc):
    df=SAMPLE[oc].copy(); name,kind=OUTCOMES[oc]
    for c in CTRL_CAT: df[c]=df[c].astype('category')
    fb={s:ftree(df,s) for s in SPECS}; y=df[oc].values; folds=[]
    selm=(roc_auc_score if kind=='clf' else r2_score)
    for sd in SEEDS:
        cv=(StratifiedKFold(N_SPLITS,shuffle=True,random_state=sd) if kind=='clf' else KFold(N_SPLITS,shuffle=True,random_state=sd))
        for tr,te in cv.split(df,y if kind=='clf' else None):
            sub=df.iloc[tr].reset_index(drop=True); ys=y[tr]
            icv=(StratifiedKFold(3,shuffle=True,random_state=sd) if kind=='clf' else KFold(3,shuffle=True,random_state=sd))
            rec=dict(seed=sd,te=te,cfg={},y=y[te],preds={},fm=None,Xte=None)
            if kind=='reg': rec['thr']=np.quantile(y[tr],WINNER_Q)
            for s in SPECS:                                    # HP tuned PER SPEC (each spec gets its own inner-CV-selected config)
                best,bv=None,-1e9
                for nm,hp in GN.items():
                    vv=[selm(ys[ite],tfit(sub[fb[s]].iloc[itr],ys[itr],sub[fb[s]].iloc[ite],kind,dict(hp))[0]) for itr,ite in icv.split(sub,ys if kind=='clf' else None)]
                    if np.mean(vv)>bv: bv,best=np.mean(vv),nm
                rec['cfg'][s]=best
                p,m=tfit(df[fb[s]].iloc[tr],y[tr],df[fb[s]].iloc[te],kind,dict(GN[best])); rec['preds'][s]=p
                if s=='+nlp': rec['fm']=m; rec['Xte']=df[fb[s]].iloc[te]
            folds.append(rec)
        print(f'  nested {oc} seed {sd} done',flush=True)
    return folds
def repeated_lin(oc):
    df=SAMPLE[oc].copy(); name,kind=OUTCOMES[oc]
    tg={c:df[c].astype(str) for c in CTRL_CAT}; dd=pd.get_dummies(df,columns=CTRL_CAT,drop_first=True,dtype=int)
    for c in CTRL_CAT: dd[c]=tg[c].values
    fb={s:flin(dd,s) for s in SPECS}; y=dd[oc].values; tech=dd['tech_field_group'].astype(str).values; folds=[]
    for sd in SEEDS:
        cv=(StratifiedKFold(N_SPLITS,shuffle=True,random_state=sd) if kind=='clf' else KFold(N_SPLITS,shuffle=True,random_state=sd))
        for tr,te in cv.split(dd,y if kind=='clf' else None):
            rec=dict(seed=sd,te=te,y=y[te],preds={})
            if kind=='reg': rec['thr']=np.quantile(y[tr],WINNER_Q)
            for s in SPECS:
                Xtr,Xte=winsor_impute(dd[fb[s]].iloc[tr],dd[fb[s]].iloc[te],tech[tr],tech[te]); sc=StandardScaler(); A=sc.fit_transform(Xtr.astype(float)); B=sc.transform(Xte.astype(float))
                if kind=='clf':
                    m=LogisticRegressionCV(Cs=10,cv=5,penalty='l2',solver='lbfgs',scoring='roc_auc',class_weight='balanced',max_iter=2000,random_state=SEED); m.fit(A,y[tr]); rec['preds'][s]=m.predict_proba(B)[:,1]
                else:
                    m=RidgeCV(alphas=np.logspace(-3,3,13),cv=5); m.fit(A,y[tr]); rec['preds'][s]=m.predict(B)
            folds.append(rec)
        print(f'  linear {oc} seed {sd} done',flush=True)
    return folds

In [6]:
CACHE=OUT/'cv_folds.pkl'
if CACHE.exists():
    TF,LF=pickle.load(open(CACHE,'rb')); print('loaded cache')
else:
    TF={oc:nested_tree(oc) for oc in OUTCOMES}; LF={oc:repeated_lin(oc) for oc in OUTCOMES}
    pickle.dump((TF,LF),open(CACHE,'wb')); print('saved cache')

  nested log_total_funding_usd seed 42 done


  nested log_total_funding_usd seed 43 done


  nested log_total_funding_usd seed 44 done


  nested log_total_funding_usd seed 45 done


  nested log_total_funding_usd seed 46 done


...

  linear exit_binary seed 48 done


  linear exit_binary seed 49 done


  linear exit_binary seed 50 done


  linear exit_binary seed 51 done


saved cache


In [7]:
def perf(folds,kind,which,mask=None):
    o={s:[] for s in SPECS}
    for r in folds:
        m=np.ones(len(r['y']),bool) if mask is None else mask[r['te']]
        if m.sum()<20: continue
        yy=r['y'][m]
        if kind=='clf' and (yy.sum()<2 or yy.sum()>len(yy)-2): continue
        for s in SPECS:
            pp=r['preds'][s][m]
            o[s].append({'auc':lambda:roc_auc_score(yy,pp),'pr':lambda:average_precision_score(yy,pp),'r2':lambda:r2_score(yy,pp),'rmse':lambda:np.sqrt(mean_squared_error(yy,pp))}[which]())
    return {s:np.array(o[s]) for s in SPECS}
def lift(folds,kind,kf,mask=None):
    o={s:[] for s in SPECS}
    for r in folds:
        m=np.ones(len(r['y']),bool) if mask is None else mask[r['te']]
        if m.sum()<20: continue
        tp=(r['y'][m]>=r['thr']).astype(int) if kind=='reg' else r['y'][m].astype(int)
        if tp.sum()<2: continue
        br=tp.mean(); k=max(1,int(round(len(tp)*kf)))
        for s in SPECS:
            order=np.argsort(-r['preds'][s][m],kind='stable')[:k]; o[s].append(tp[order].mean()/br if br>0 else np.nan)
    return {s:np.array(o[s]) for s in SPECS}
def ct(per,ref,alt): c=nb_one(per[alt]-per[ref]); return c, per[ref].mean(), per[alt].mean()
HB={'r2':True,'rmse':False,'auc':True,'pr':True}  # metric direction (RMSE is lower-better)
def contrast(per,ref,alt,metric):
    a,b=per[ref],per[alt]
    if HB[metric]:
        c=nb_one(b-a); ci=f"[{c['lb']:+.3f}, +inf)"
    else:
        c=nb_one(a-b); ci=f"(-inf, {-c['lb']:+.3f}]"
    return b.mean()-a.mean(), ci, c['p'], b.mean()
SP={'controls':'Controls','nlp_only':'Controls + NLP','baseline':'Controls + Traditional','+nlp':'Full model'}
def show(df):
    from IPython.display import display; display(df)

## Table 3 — Cross-validated predictive performance (gradient-boosted tree, nested CV)

In [8]:
rows=[]
# Exit (classification): ROC-AUC / PR-AUC
ea=perf(TF['exit_binary'],'clf','auc'); ep=perf(TF['exit_binary'],'clf','pr')
for s in SPECS: rows.append(['Exit',SP[s],f"{ea[s].mean():.3f}",f"{ep[s].mean():.3f}"])
# Funding (regression): R2 / RMSE
fr=perf(TF['log_total_funding_usd'],'reg','r2'); fm=perf(TF['log_total_funding_usd'],'reg','rmse')
for s in SPECS: rows.append(['Cumulative funding',SP[s],f"{fr[s].mean():.3f}",f"{fm[s].mean():.3f}"])
T3=pd.DataFrame(rows,columns=['Outcome','Specification','ROC-AUC / R²','PR-AUC / RMSE'])
T3.to_csv(OUT/'table3_performance.csv',index=False); show(T3)

,Outcome,Specification,ROC-AUC / R²,PR-AUC / RMSE
0,Exit,Controls,0.669,0.335
1,Exit,Controls + NLP,0.663,0.335
2,Exit,Controls + Traditional,0.676,0.353
3,Exit,Full model,0.672,0.352
4,Cumulative funding,Controls,0.077,1.800
5,Cumulative funding,Controls + NLP,0.093,1.784
6,Cumulative funding,Controls + Traditional,0.135,1.742
7,Cumulative funding,Full model,0.140,1.737


## Table 4 — H1: incremental value of NLP over the traditional block (one-sided NB, oriented toward improvement)

In [9]:
rows=[]
for oc,(lab,kind) in OUTCOMES.items():
    mets=[('R²','r2'),('RMSE','rmse')] if kind=='reg' else [('ROC-AUC','auc'),('PR-AUC','pr')]
    for nm,w in mets:
        nd,ci,p,_=contrast(perf(TF[oc],kind,w),'baseline','+nlp',w)
        rows.append([lab,nm,f"{nd:+.4f}",ci,f"{p:.3f}{stt(p)}"])
T4=pd.DataFrame(rows,columns=['Outcome','Metric','Δ (full−baseline)','95% one-sided CI (NB)','NB p (1-sided, improvement)'])
T4.to_csv(OUT/'table4_headline_contrast.csv',index=False); show(T4)

,Outcome,Metric,Δ (full−baseline),95% one-sided CI (NB),"NB p (1-sided, improvement)"
0,Cumulative funding,R²,+0.0051,"[-0.007, +inf)",0.249
1,Cumulative funding,RMSE,-0.0051,"(-inf, +0.007]",0.245
2,Exit,ROC-AUC,-0.0037,"[-0.013, +inf)",0.745
3,Exit,PR-AUC,-0.0009,"[-0.014, +inf)",0.547


## Table 14 — Block-vs-controls decomposition (Controls → +block), one-sided, both families, both metrics per outcome (R²/RMSE, ROC-AUC/PR-AUC)

In [10]:
METR={'reg':[('R²','r2'),('RMSE','rmse')],'clf':[('ROC-AUC','auc'),('PR-AUC','pr')]}
rows=[]
for oc,(lab,kind) in OUTCOMES.items():
    for mlab,w in METR[kind]:
        for fam,F in [('tree',TF[oc]),('linear',LF[oc])]:
            per=perf(F,kind,w); rows.append([lab,mlab,fam,'Controls (ref)',f"{per['controls'].mean():.3f}",'','',''])
            for s,n in [('baseline','+ Traditional'),('nlp_only','+ NLP'),('+nlp','+ Full block')]:
                nd,ci,p,altm=contrast(per,'controls',s,w)
                rows.append([lab,mlab,fam,n,f"{altm:.3f}",f"{nd:+.3f}",ci,f"{p:.3f}{stt(p)}"])
T14=pd.DataFrame(rows,columns=['Outcome','Metric','Family','Spec','Held-out fit','Δ vs Controls','95% 1-sided CI','NB p (1-sided, improvement)'])
T14.to_csv(OUT/'table14_block_contrasts.csv',index=False); show(T14)

,Outcome,Metric,Family,Spec,Held-out fit,Δ vs Controls,95% 1-sided CI,"NB p (1-sided, improvement)"
0,Cumulative funding,R²,tree,Controls (ref),0.077,,,
1,Cumulative funding,R²,tree,+ Traditional,0.135,+0.058,"[+0.035, +inf)",0.000***
2,Cumulative funding,R²,tree,+ NLP,0.093,+0.016,"[+0.002, +inf)",0.032**
3,Cumulative funding,R²,tree,+ Full block,0.140,+0.063,"[+0.040, +inf)",0.000***
4,Cumulative funding,R²,linear,Controls (ref),0.049,,,
5,Cumulative funding,R²,linear,+ Traditional,0.093,+0.044,"[+0.031, +inf)",0.000***
6,Cumulative funding,R²,linear,+ NLP,0.072,+0.023,"[+0.012, +inf)",0.000***
7,Cumulative funding,R²,linear,+ Full block,0.102,+0.053,"[+0.040, +inf)",0.000***
8,Cumulative funding,RMSE,tree,Controls (ref),1.800,,,
9,Cumulative funding,RMSE,tree,+ Traditional,1.742,-0.058,"(-inf, -0.034]",0.000***


## Table 6 — Heterogeneity (H2 portfolio size, H3 technology field), tree, Traditional→Full (one-sided)

In [11]:
def het(oc,folds):
    lab,kind=OUTCOMES[oc]; w='r2' if kind=='reg' else 'auc'; mk=MASK[oc]; out=[]
    for nm,mask in [('Small (≤3 patents)',mk['small']),('Large (≥4 patents)',~mk['small']),('Life sciences',mk['life']),('Other fields',~mk['life'])]:
        c,rm,_=ct(perf(folds,kind,w,mask=mask),'baseline','+nlp'); out.append([lab,nm,f"{rm:.3f}",f"{c['d']:+.3f}",f"{c['p']:.3f}{stt(c['p'])}"])
    return out
rows=[r for oc in OUTCOMES for r in het(oc,TF[oc])]
T6=pd.DataFrame(rows,columns=['Outcome','Stratum','Trad. fit','Δ (+NLP)','NB p (1-sided)'])
T6.to_csv(OUT/'table6_heterogeneity_tree.csv',index=False); show(T6)

,Outcome,Stratum,Trad. fit,Δ (+NLP),NB p (1-sided)
0,Cumulative funding,Small (≤3 patents),0.082,+0.012,0.099*
1,Cumulative funding,Large (≥4 patents),0.112,-0.014,0.824
2,Cumulative funding,Life sciences,0.177,+0.004,0.328
3,Cumulative funding,Other fields,0.064,+0.006,0.278
4,Exit,Small (≤3 patents),0.668,-0.008,0.890
5,Exit,Large (≥4 patents),0.673,+0.003,0.390
6,Exit,Life sciences,0.684,-0.002,0.609
7,Exit,Other fields,0.670,-0.006,0.781


## Table 5 — Top features by mean |SHAP| (full model)

In [12]:
def shap_tab(folds):
    acc={}
    for r in folds:
        sv=shap.TreeExplainer(r['fm']).shap_values(r['Xte'])
        if isinstance(sv,list): sv=sv[1]
        for f,v in zip(r['Xte'].columns,np.abs(sv).mean(axis=0)):
            if not f.endswith('_missing'): acc.setdefault(f,[]).append(v)
    return sorted(((LABEL.get(f,f),np.mean(v)) for f,v in acc.items()),key=lambda x:-x[1])
rows=[[OUTCOMES[oc][0],i+1,f,round(v,4)] for oc in OUTCOMES for i,(f,v) in enumerate(shap_tab(TF[oc]))]
T5=pd.DataFrame(rows,columns=['Outcome','Rank','Feature','Mean |SHAP|'])
T5.to_csv(OUT/'table5_shap.csv',index=False); show(T5)

,Outcome,Rank,Feature,Mean |SHAP|
0,Cumulative funding,1,Family size,0.2506
1,Cumulative funding,2,Portfolio size (n_patents),0.2353
2,Cumulative funding,3,Novelty,0.2007
3,Cumulative funding,4,Country region,0.1630
4,Cumulative funding,5,Patent scope,0.1624
5,Cumulative funding,6,NPL citations,0.1557
6,Cumulative funding,7,Age at first funding,0.0646
7,Cumulative funding,8,Claims,0.0596
8,Cumulative funding,9,Competitive density,0.0579
9,Cumulative funding,10,Radicalness,0.0507


## Figure 1 — Top-10% screening lift; downward whisker = one-sided 95% NB lower bound (interval → +∞). Funding 'winner' = top-decile funding.

In [13]:
def se_nb(x):
    x=np.asarray(x,float); n=len(x); return np.sqrt((1/n+RHO)*x.var(ddof=1))
t95=stats.t.ppf(0.95,N_SEEDS*N_SPLITS-1)
fig,ax=plt.subplots(1,2,figsize=(9,3.6)); plt.rcParams.update({'font.family':'serif'})
for j,(oc,(lab,kind)) in enumerate(OUTCOMES.items()):
    per=lift(TF[oc],kind,0.10); means=[per[s].mean() for s in SPECS]; loerr=[t95*se_nb(per[s]) for s in SPECS]
    ax[j].bar(range(4),means,yerr=[loerr,[0]*4],color='white',edgecolor='black',capsize=3); ax[j].axhline(1,ls=':',c='gray')
    ax[j].set_xticks(range(4)); ax[j].set_xticklabels(['Controls','+NLP','+Trad','Full'],fontsize=8)
    ax[j].set_title(f'({chr(97+j)}) {lab}',fontsize=10); ax[j].set_ylabel('Lift at Top-10%\n(whisker = 1-sided 95% LB)' if j==0 else '')
fig.tight_layout(); fig.savefig(OUT/'lift_fig.png',dpi=150); plt.show()

## Table 12 / 13 — Penalised-linear performance and heterogeneity (App 6.6)

In [14]:
rows=[]
le_a=perf(LF['exit_binary'],'clf','auc'); le_p=perf(LF['exit_binary'],'clf','pr')
for s in SPECS: rows.append(['Exit',SP[s],f"{le_a[s].mean():.3f}",f"{le_p[s].mean():.3f}"])
lf_r=perf(LF['log_total_funding_usd'],'reg','r2'); lf_m=perf(LF['log_total_funding_usd'],'reg','rmse')
for s in SPECS: rows.append(['Cumulative funding',SP[s],f"{lf_r[s].mean():.3f}",f"{lf_m[s].mean():.3f}"])
T12=pd.DataFrame(rows,columns=['Outcome','Specification','ROC-AUC / R²','PR-AUC / RMSE'])
T12.to_csv(OUT/'table12_linear_performance.csv',index=False); show(T12)
rows=[r for oc in OUTCOMES for r in het(oc,LF[oc])]
T13=pd.DataFrame(rows,columns=['Outcome','Stratum','Trad. fit','Δ (+NLP)','NB p (1-sided)'])
T13.to_csv(OUT/'table13_linear_heterogeneity.csv',index=False); show(T13)

,Outcome,Specification,ROC-AUC / R²,PR-AUC / RMSE
0,Exit,Controls,0.678,0.343
1,Exit,Controls + NLP,0.681,0.357
2,Exit,Controls + Traditional,0.679,0.352
3,Exit,Full model,0.679,0.356
4,Cumulative funding,Controls,0.049,1.827
5,Cumulative funding,Controls + NLP,0.072,1.805
6,Cumulative funding,Controls + Traditional,0.093,1.784
7,Cumulative funding,Full model,0.102,1.775


,Outcome,Stratum,Trad. fit,Δ (+NLP),NB p (1-sided)
0,Cumulative funding,Small (≤3 patents),0.053,+0.014,0.034**
1,Cumulative funding,Large (≥4 patents),0.022,-0.004,0.635
2,Cumulative funding,Life sciences,0.104,+0.014,0.039**
3,Cumulative funding,Other fields,0.048,+0.005,0.290
4,Exit,Small (≤3 patents),0.665,-0.003,0.680
5,Exit,Large (≥4 patents),0.681,+0.008,0.143
6,Exit,Life sciences,0.683,+0.002,0.387
7,Exit,Other fields,0.676,-0.002,0.672


## Table 17 — Full screening-lift (tree, top-10%): three NB-corrected one-sided lift contrasts (Traditional vs Controls; NLP vs Controls; Full vs Traditional = H1), pooled and within each stratum.

In [15]:
CONTR=[('Traditional vs Controls','controls','baseline'),('NLP vs Controls','controls','nlp_only'),('Full vs Traditional (H1)','baseline','+nlp')]
rows=[]
for oc,(lab,kind) in OUTCOMES.items():
    mk=MASK[oc]
    for snm,mask in [('Pooled',None),('Small (≤3)',mk['small']),('Large (≥4)',~mk['small']),('Life sciences',mk['life']),('Other fields',~mk['life'])]:
        per=lift(TF[oc],kind,0.10,mask=mask)
        for cnm,ref,alt in CONTR:
            c=nb_one(per[alt]-per[ref])
            rows.append([lab,snm,cnm,round(per[ref].mean(),2),round(per[alt].mean(),2),f"{c['d']:+.2f}",f"{c['p']:.2f}{stt(c['p'])}"])
T17=pd.DataFrame(rows,columns=['Outcome','Stratum','Contrast','Lift ref','Lift alt','Δ','NB p (1-sided)'])
T17.to_csv(OUT/'table17_lift.csv',index=False); show(T17)

,Outcome,Stratum,Contrast,Lift ref,Lift alt,Δ,NB p (1-sided)
0,Cumulative funding,Pooled,Traditional vs Controls,1.91,2.67,+0.76,0.01***
1,Cumulative funding,Pooled,NLP vs Controls,1.91,2.16,+0.25,0.17
2,Cumulative funding,Pooled,Full vs Traditional (H1),2.67,2.68,+0.01,0.48
3,Cumulative funding,Small (≤3),Traditional vs Controls,2.43,2.75,+0.32,0.30
4,Cumulative funding,Small (≤3),NLP vs Controls,2.43,2.65,+0.21,0.37
5,Cumulative funding,Small (≤3),Full vs Traditional (H1),2.75,2.93,+0.17,0.31
6,Cumulative funding,Large (≥4),Traditional vs Controls,2.08,2.32,+0.25,0.35
7,Cumulative funding,Large (≥4),NLP vs Controls,2.08,1.89,-0.19,0.63
8,Cumulative funding,Large (≥4),Full vs Traditional (H1),2.32,2.42,+0.09,0.42
9,Cumulative funding,Life sciences,Traditional vs Controls,2.13,2.62,+0.49,0.13


## App 6.11 — Variance-Inflation Factors (funding sample, standardised predictors)

In [16]:
base=SAMPLE['log_total_funding_usd']; dl=pd.get_dummies(base,columns=CTRL_CAT,drop_first=True,dtype=int)
cont=['founding_year','n_patents','age_at_first_funding']+TRAD+NLP; oh=[c for c in dl.columns if any(c.startswith(p+'_') for p in CTRL_CAT)]
Xv=dl[cont+oh].copy()
for c in TRAD+NLP: Xv[c]=Xv[c].fillna(0)
Xv=((Xv-Xv.mean())/Xv.std(ddof=0)).dropna(axis=1)
def vif(cols):
    M=Xv[cols].values; o={}
    for j,cn in enumerate(cols):
        yj=M[:,j]; Xo=np.column_stack([np.ones(len(yj)),np.delete(M,j,axis=1)]); b,_,_,_=np.linalg.lstsq(Xo,yj,rcond=None)
        r2=1-np.sum((yj-Xo@b)**2)/np.sum((yj-yj.mean())**2); o[cn]=1/(1-r2) if r2<1 else np.inf
    return o
vf=vif(list(Xv.columns)); VIF=pd.DataFrame([[LABEL.get(c,c),round(vf[c],2)] for c in Xv.columns],columns=['Predictor','VIF'])
VIF.to_csv(OUT/'vif.csv',index=False); show(VIF); print('Max VIF:',round(max(vf.values()),2))

,Predictor,VIF
0,Founding year,1.19
1,Portfolio size (n_patents),1.10
2,Age at first funding,1.18
3,Backward citations,1.04
4,NPL citations,1.27
5,Originality,2.41
6,Radicalness,1.65
7,Claims,1.23
8,Family size,1.36
9,Patent scope,1.54


Max VIF: 3.71


## App 6.12 — Power / minimum detectable effect (80% power, one-sided α=.05)

In [17]:
dd=N_SEEDS*N_SPLITS-1; fac=stats.t.ppf(0.95,dd)+stats.t.ppf(0.80,dd); rows=[]
for oc,(lab,kind) in OUTCOMES.items():
    w='r2' if kind=='reg' else 'auc'; per=perf(TF[oc],kind,w)
    for ref,alt,nm in [('controls','baseline','Controls→Traditional'),('controls','nlp_only','Controls→NLP'),('baseline','+nlp','Traditional→Full (H1)')]:
        c,_,_=ct(per,ref,alt); rows.append([lab,nm,f"{c['d']:+.4f}",round(c['se'],4),round(fac*c['se'],4)])
POW=pd.DataFrame(rows,columns=['Outcome','Contrast','observed Δ','NB SE','MDE@80% (1-sided)'])
POW.to_csv(OUT/'power_mde.csv',index=False); show(POW)

,Outcome,Contrast,observed Δ,NB SE,MDE@80% (1-sided)
0,Cumulative funding,Controls→Traditional,+0.0579,0.0138,0.0348
1,Cumulative funding,Controls→NLP,+0.0164,0.0087,0.0219
2,Cumulative funding,Traditional→Full (H1),+0.0051,0.0074,0.0188
3,Exit,Controls→Traditional,+0.0066,0.0114,0.0289
4,Exit,Controls→NLP,-0.0064,0.0074,0.0186
5,Exit,Traditional→Full (H1),-0.0037,0.0056,0.0141


## Notes
Tree estimates use nested CV; the linear model is the penalised-regression family. Inference is
one-sided Nadeau–Bengio. CSV tables, the cached fold predictions (`cv_folds.pkl`), and the lift
figure (`lift_fig.png`) are written to `data/processed/ml_results/`.